# Qiskit Fall Fest 2026 — Phase 1
## S3: Noise in Trotter Simulation

This notebook follows the supplied Phase-1 architecture workflow. All standalone numerical settings are explicitly illustrative.

## 1. How to use this notebook
Run top-to-bottom after installing `requirements-practice.txt` for the self-contained path, or the full requirements for the optional Qiskit path.

## 2. Phase 1 overview
Scope: one problem statement (S3), Processor A/B comparison, architecture/resource metrics, reproducible evidence, and reporting. No Phase-2 processor is included.

## 3. Repository structure
Key paths: `src/`, `processors/`, `results/`, `report/`, `tests/`, and `scripts/`.

## 4. Core mental model
Problem/algorithm → logical circuit → placement/routing/gate decomposition → physical processor → measurement → problem + architecture metrics.

## 5. Quantum foundations
The architecture study uses Bell-state correlations as an interpretable coherence benchmark and the S3 demonstrator uses a small Trotter product formula.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd()
config = json.loads((ROOT / 'data' / 'practice_config.json').read_text())
config

## 6. Processor as a graph
A processor is represented by physical qubits plus a coupling map. Nonadjacent logical interactions require routing and can add depth and two-qubit work.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

for label in ['A','B']:
    p = config['processors'][label]
    G = nx.Graph()
    G.add_nodes_from(range(config['system']['n_qubits']))
    G.add_edges_from(tuple(e) for e in p['coupling_map'])
    nx.draw(G, with_labels=True)
    plt.title(f'Practice Processor {label}: {p["name"]}')
    plt.show()

## 7. Bell-state example
The supplied guide emphasizes XX, YY and ZZ correlations rather than relying only on computational-basis agreement.

In [ ]:
from src.bell_benchmark import bell_state, bell_correlations, fidelity_to_bell

state = bell_state()
print('Correlations:', bell_correlations(state))
print('Ideal fidelity:', fidelity_to_bell(__import__('numpy').outer(state, state.conj())))

## 8. Routing and SWAP overhead
Processor B uses a line-like topology. A long-range logical interaction therefore needs routing, increasing two-qubit operations.

In [ ]:
from src.practice_s3 import logical_trotter_circuit, PracticeProcessor, compile_practice

s = config['system']
logical = logical_trotter_circuit(s['trotter_steps'], s['total_time'], s['J'], s['h'])
models = {}
for key in ['A','B']:
    p = config['processors'][key]
    models[key] = PracticeProcessor(p['name'], s['n_qubits'], tuple(tuple(e) for e in p['coupling_map']), p['p1'], p['p2'], p['readout_error'])
compiled = {k: compile_practice(logical, v) for k,v in models.items()}
print('Logical gates:', len(logical))
print('Compiled A:', len(compiled['A']))
print('Compiled B:', len(compiled['B']))

## 9. Transpilation
The optional Qiskit adapter maps the same logical circuit to a processor coupling map. The runnable NumPy path keeps this repository executable without requiring Qiskit.

In [ ]:
# Optional Qiskit path
try:
    from qiskit import QuantumCircuit
    print('Qiskit available:', True)
except ImportError:
    print('Qiskit available:', False, '- use the practice path')

## 10. Noise
The standalone demonstrator uses a stochastic Pauli-error proxy. It is an illustrative model, not a characterization of a physical device.

## 11. Required measurements
Record fidelity/quality together with depth, two-qubit gates, routing/SWAP overhead, physical qubits and uncertainty.

## 12. Processor definitions
Processor settings are stored as machine-readable JSON under `processors/`. They are project-defined illustrative comparators.

## 13. Problem statement
**S3 — Noise in Trotter Simulation** is the single selected problem statement for this repository.

## 14. A/B experiment
Use the same logical workload and compare it on Processor A and Processor B.

In [ ]:
from src.practice_s3 import ideal_exact_state, benchmark_processor

exact = ideal_exact_state(s['n_qubits'], s['total_time'], s['J'], s['h'])
rows = []
for i,key in enumerate(['A','B']):
    row = benchmark_processor(models[key], logical, exact, s['n_qubits'], int(config['shots_or_trajectories']), int(config['seed']) + i)
    row['processor'] = key
    row['processor_name'] = models[key].name
    rows.append(row)
rows

## 15. A/B comparison
The comparison below emphasizes architecture-induced cost and resulting noisy quality.

In [ ]:
import pandas as pd
comparison = pd.DataFrame(rows)
display(comparison[['processor','processor_name','mean_fidelity','depth','two_qubit_gate_count','swap_count','physical_qubits']])

## 16. Protection / error detection
The guide identifies parity/stabilizer checks and postselection as possible directions. The Bell benchmark records postselection yield as an additional observable.

## 17. Scientific reporting
Report processor definition, circuit version, shots/trajectories, seed, noise assumptions, depth, two-qubit gates, routing/SWAPs, qubit use, postselection yield and uncertainty.

## 18. Results and figures
Numerical outputs are stored in `results/tables/`; figures can be regenerated by the scripts in `scripts/`.

## 19. Experience-level approaches
Explorer: Bell + simple error detection. Builder: placement/routing and checks. Research: geometry-aware encodings or stabilizer networks. This repository uses the Builder-oriented architecture study plus a compact S3 demonstrator.

## 20. Workflow checklist
One PS; same workload on A/B; measured PS and architecture metrics; graphs/tables; README/report; environment; tests; reproducibility; AI disclosure.

## 21. Final experiment structure
The Phase-1 endpoint is the A/B comparison and evidence-based explanation.

## 22. Submission checklist
Verify repository accessibility, clean setup, required files, machine-readable results, reproducible execution and documented limitations.

## 23. Participant lab notes
Problem: S3. Method: Trotter + architecture/noise study. Processor A/B: illustrative definitions. Evidence: saved tables, figures and tests. Limitations: no unique official S3 numerical instance is defined by the supplied guide.

In [ ]:
print('Notebook workflow complete.')